In [ ]:
!if [ -d diffusion-inpainting-3d ]; then
    cd diffusion-inpainting-3d && git pull;
else
    git clone https://github.com/gCattt/diffusion-inpainting-3d.git;
fi

%cd diffusion-inpainting-3d

In [ ]:
# install base deps without torch/torchvision/pytorch3d CPU-only
# furthermore there is no need for pytorch3d without the rendering phase
!grep -vE '^(torch|torchvision|pytorch3d)=' requirements.txt > requirements-colab.txt
!pip install -r requirements-colab.txt --quiet --no-cache-dir
!pip install xformers --quiet --no-cache-dir

In [ ]:
import torch
#print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rsync -av --delete /content/drive/MyDrive/diffusion-inpainting-3d-data/ data/

In [ ]:
import yaml
from pathlib import Path

p = Path("configs/inpainting_config.yaml")

cfg = yaml.safe_load(p.read_text())
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
p.write_text(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
from src.inpainting.batch_inpaint import inpaint_main

inpaint_main("configs/inpainting_config.yaml")

In [ ]:
from src.texture.backprojection import backprojection_main

backprojection_main()

In [ ]:
!rsync -av --delete data/outputs/ /content/drive/MyDrive/diffusion-inpainting-3d-data/outputs/